# 🚀 MASTER GOOGLE COLAB: HUẤN LUYỆN 3 MÔ HÌNH TRANSFORMER (6 EXPERIMENTS)

**Mục tiêu:** Huấn luyện toàn bộ 6 thí nghiệm Transformer cho cả 2 tác vụ trên GPU Colab/Kaggle:
1. **PhoBERT** (`vinai/phobert-base-v2`): Task 1 & Task 2 (có tiền xử lý tách từ PyVi)
2. **ViDeBERTa** (`Fsoft-AIC/videberta-base`): Task 1 & Task 2
3. **XLM-RoBERTa** (`FacebookAI/xlm-roberta-base`): Task 1 & Task 2

**Các tính năng chuẩn hóa:**
- Gradient Accumulation (Physical Batch 16 x 2 steps = Effective Batch 32)
- Mixed Precision (AMP fp16)
- Dev Macro-F1 Early Stopping (Patience = 2)
- Khóa tập Test (Chỉ đánh giá 1 lần duy nhất trên Best Dev Checkpoint)
- Tự động đóng gói tất cả artifacts thành file `.zip` để tải về.

## 1. Cài đặt Môi trường & Dependencies

In [ ]:
!pip install -q transformers==4.49.0 datasets==3.3.2 pyvi==0.1.1 accelerate==1.4.0 scikit-learn==1.6.1 seaborn==0.13.2

## 2. Kiểm tra GPU

In [ ]:
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('Memory Allocated:', round(torch.cuda.memory_allocated(0)/1024**3, 1), 'GB')
else:
    print('WARNING: GPU not detected! Hãy đổi Runtime sang GPU T4 trước khi chạy!')

## 3. Khởi tạo Đường dẫn & Import Engine

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from train import train_transformer_task
from transformer_model import run_all_smoke_tests
from aggregate_results import aggregate_all

print('Project Root:', PROJECT_ROOT)

## 4. Chạy Smoke Test xác thực trước khi Train

In [ ]:
smoke_res = run_all_smoke_tests(device='cuda' if torch.cuda.is_available() else 'cpu')
print('Smoke Test Results Summary:')
for k, v in smoke_res.items():
    print(f'  * {k}: {v.get("status", "UNKNOWN")}')

## 5. Chạy Huấn luyện 6 Thí nghiệm Transformer

In [ ]:
models_to_train = ['phobert', 'videberta', 'xlmr']
tasks_to_train = ['task1', 'task2']

results_summary = []
for m_key in models_to_train:
    for t_key in tasks_to_train:
        print(f'\n>>>>>>>>>>>>>>>>> [START] MODEL: {m_key.upper()} | TASK: {t_key.upper()} <<<<<<<<<<<<<<<<<')
        res = train_transformer_task(
            model_key=m_key,
            task_name=t_key,
            epochs=5,
            batch_size=16,
            grad_accum_steps=2,
            lr=2e-5,
            patience=2,
            seed=42,
            max_length=128
        )
        results_summary.append(res)
        print(f'>>>>>>>>>>>>>>>>> [COMPLETED] {m_key.upper()} - {t_key.upper()} <<<<<<<<<<<<<<<<<\n')

## 6. Tổng hợp Kết quả Benchmark Toàn diện & Vẽ Biểu đồ

In [ ]:
aggregate_all()
print('\n--- TỔNG HỢP KẾT QUẢ TÁC VỤ 1 ---')
display(pd.read_csv(PROJECT_ROOT / 'results' / 'test_results_task1.csv'))

print('\n--- TỔNG HỢP KẾT QUẢ TÁC VỤ 2 ---')
display(pd.read_csv(PROJECT_ROOT / 'results' / 'test_results_task2.csv'))

## 7. Đóng gói Artifacts thành file ZIP để tải về máy

In [ ]:
import shutil

zip_output = PROJECT_ROOT / 'colab_artifacts_export'
dirs_to_zip = ['results', 'artifacts', 'figures', 'report']

temp_export = PROJECT_ROOT / 'export_bundle'
temp_export.mkdir(parents=True, exist_ok=True)
for d in dirs_to_zip:
    src_d = PROJECT_ROOT / d
    if src_d.exists():
        shutil.copytree(src_d, temp_export / d, dirs_exist_ok=True)

shutil.make_archive(str(zip_output), 'zip', temp_export)
shutil.rmtree(temp_export)
print(f'PASS: Đã đóng gói thành công file {zip_output}.zip! Hãy tải file này về và giải nén đè vào Model_final_v3.')